# Googleドライブのいきものクイズを起動する

先に `migrate_to_drive.ipynb` で一度だけ移行してください。このノートブックはGoogleドライブへ保存して使います。

1. 最初のセルを実行し、Googleドライブへの接続を許可します。
2. 「準備完了」が出てから起動セルを実行します。
3. 今回表示された新しい `gradio.live` のURLを開きます。遊んでいる間は起動セルを動かしたままにします。

コード・画像の原本は **マイドライブ／kinoko_quiz／kinoko-quiz** です。毎回ドライブの現在の内容を読み込み、GitHubからは取得しません。ライブラリのインストールとGradioの共有URLにはインターネット接続が必要です。

変更する場合は起動セルを停止し、ドライブの原本を編集してから、最初のセルから実行し直してください。Colabの一時コピーへの編集はドライブに保存されません。


In [ ]:
from google.colab import drive
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

SETUP_COMPLETE = False
drive.mount('/content/drive')
# 移行先を変更した場合は、ここも同じフォルダに変更してください。
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/kinoko_quiz/kinoko-quiz')
helper = DRIVE_PROJECT_DIR / 'colab' / 'drive_files.py'
if not helper.is_file():
    raise FileNotFoundError(f'移行済みのゲームがありません: {DRIVE_PROJECT_DIR}。先に移行ノートブックを実行してください。')
spec = importlib.util.spec_from_file_location('quiz_drive_files', helper)
drive_files = importlib.util.module_from_spec(spec)
spec.loader.exec_module(drive_files)

print('1/2：ドライブの現在のコード・画像を読み込んでいます。', flush=True)
project_dir = drive_files.prepare_local_project(DRIVE_PROJECT_DIR, Path('/content'))
print('2/2：必要なライブラリをインストールしています。', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
               cwd=project_dir, check=True)
SETUP_COMPLETE = True
print(f'準備完了：Googleドライブ {DRIVE_PROJECT_DIR}。次の起動セルを実行してください。', flush=True)


In [ ]:
# このセルを動かしたまま、新しく表示されたURLを開きます。
if not globals().get('SETUP_COMPLETE', False):
    raise RuntimeError('最初のセルを実行し、「準備完了」を待ってください。')
subprocess.run([sys.executable, 'app.py'], cwd=project_dir,
               env={**os.environ, 'GRADIO_SHARE': 'true'}, check=True)
